<a href="https://colab.research.google.com/github/DANIEL-VELASCO/taller1-deep-learning/blob/main/code/punto2_cnn_fashion_mnist/03_bono_redes_modernas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Punto 2 · Notebook 03 – Bono: EfficientNetV2 y ConvNeXt

**Taller 1 – Aprendizaje Profundo** · Maestría en Inteligencia Artificial, Pontificia Universidad Javeriana

El bono pide extender el análisis a una red moderna (EfficientNetV2, ConvNeXt o ResNeXt). Tomamos dos,
porque representan las dos ideas que siguieron a VGG16 y MobileNetV2:

- **EfficientNetV2-B0** (2021): la línea de MobileNetV2 llevada más lejos. Bloques MBConv y Fused-MBConv,
  escalado compuesto de profundidad, ancho y resolución, y entrenamiento progresivo. ~6 M de parámetros.
- **ConvNeXt-Tiny** (2022): una CNN "pura" rediseñada con las decisiones de los Transformers de visión:
  kernels depthwise de 7×7, LayerNorm, GELU y bloques de cuello invertido. ~28 M de parámetros.

Para que la comparación con las tres redes del notebook `01` sea justa, repetimos exactamente sus
condiciones: misma partición estratificada 54 000 / 6 000 / 10 000 con `SEED = 42`, mismos lotes de
128, misma entrada de 28×28×1 que se redimensiona a 96×96 dentro del modelo, mismos callbacks, misma
rejilla de búsqueda que usamos para VGG16 y MobileNetV2, misma fase de fine-tuning con learning rate
diez veces menor, y mismas métricas sobre test. También en `float32`, como quedó la corrida original.

El notebook tiene dos partes:

1. **Entrenamiento** (secciones 1 a 4). Necesita GPU. Va guardando todo en `results/punto2/bono/` a
   medida que avanza y, si Colab se desconecta, al volver a correrlo retoma donde iba.
2. **Figuras y tablas** (secciones 5 a 11). Solo lee archivos: con `ENTRENAR = False` se puede correr
   sin GPU y en segundos para ajustar una gráfica. Junta los resultados nuevos con los del notebook `01`.

Duración esperada en una T4 con `PRESUPUESTO = "completo"`: entre una y dos horas. ConvNeXt se lleva
la mayor parte.

## 0. Preparación del entorno

In [ ]:
import gc
import json
import os
import random
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,
                             log_loss, precision_recall_fscore_support)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# False: se salta el entrenamiento y solo se arman las figuras con lo que ya está en disco.
ENTRENAR = True

SEED = 42
IMG_SIZE = 96
NUM_CLASSES = 10
MODELOS_BONO = ["efficientnetv2b0", "convnext_tiny"]

# Los mismos tres presupuestos del notebook 01. Los números del informe salieron de "completo".
PRESUPUESTO = "completo"     # "completo" | "acotado" | "depuracion"
PRESUPUESTOS = {
    "completo":   {"n_busqueda": None,  "epochs_search": 12, "epochs_final": 30, "epochs_ft": 15, "bootstrap": 1000},
    "acotado":    {"n_busqueda": 15000, "epochs_search": 8,  "epochs_final": 20, "epochs_ft": 8,  "bootstrap": 1000},
    "depuracion": {"n_busqueda": 12000, "epochs_search": 4,  "epochs_final": 8,  "epochs_ft": 3,  "bootstrap": 300},
}
_P = PRESUPUESTOS[PRESUPUESTO]
N_BUSQUEDA = _P["n_busqueda"]
EPOCHS_SEARCH = _P["epochs_search"]
EPOCHS_FINAL = _P["epochs_final"]
EPOCHS_FT = _P["epochs_ft"]
N_BOOTSTRAP = _P["bootstrap"]

if ENTRENAR:
    import tensorflow as tf

    # La corrida del notebook 01 quedó en float32 (ver entorno.json). Lo fijamos igual aquí para
    # que los tiempos se puedan comparar.
    tf.keras.mixed_precision.set_global_policy("float32")
    HAY_GPU = bool(tf.config.list_physical_devices("GPU"))
    BATCH_SIZE = 128 if HAY_GPU else 64
    AUTOTUNE = tf.data.AUTOTUNE
    print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
    if not HAY_GPU:
        print("AVISO: sin GPU esto tarda horas. Entorno de ejecución > Cambiar tipo de entorno > T4.")


def seed_all(s=SEED):
    os.environ["PYTHONHASHSEED"] = str(s)
    random.seed(s)
    np.random.seed(s)
    if ENTRENAR:
        tf.random.set_seed(s)


seed_all()
print(f"PRESUPUESTO: {PRESUPUESTO} | búsqueda: {N_BUSQUEDA or 54000} imágenes x {EPOCHS_SEARCH} épocas"
      f" | final: {EPOCHS_FINAL} épocas + {EPOCHS_FT} de fine-tuning")
if PRESUPUESTO == "depuracion":
    print("OJO: presupuesto de depuración. Estos números NO sirven para el informe.")

In [ ]:
# En Colab clonamos el repo: de ahí salen los resultados del notebook 01 y ahí dejamos los del bono.
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not Path("/content/taller1-deep-learning").exists():
        !git clone -q https://github.com/DANIEL-VELASCO/taller1-deep-learning.git /content/taller1-deep-learning
    RAIZ = Path("/content/taller1-deep-learning")
else:
    RAIZ = Path.cwd().resolve().parents[1]

RUTA_P2 = RAIZ / "results" / "punto2"
RUTA_BONO = RUTA_P2 / "bono"
RUTA_BONO.mkdir(parents=True, exist_ok=True)
print("resultados del notebook 01 ->", RUTA_P2)
print("resultados del bono        ->", RUTA_BONO)

## 1. Datos: la misma partición del notebook 01

Repetimos la carga y la partición con las mismas funciones y la misma semilla, así que el
entrenamiento, la validación y el test son exactamente los mismos. Comprobamos los tamaños y el
conteo por clase como verificación.

In [ ]:
if ENTRENAR:
    (x_dev, y_dev), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
    x_train, x_val, y_train, y_val = train_test_split(
        x_dev, y_dev, test_size=0.1, random_state=SEED, stratify=y_dev)
    assert (len(x_train), len(x_val), len(x_test)) == (54000, 6000, 10000)
    assert (np.bincount(y_val) == 600).all() and (np.bincount(y_test) == 1000).all()
    print("train", x_train.shape, "| val", x_val.shape, "| test", x_test.shape)

    def _a_float32(imagenes, etiquetas):
        return tf.cast(imagenes, tf.float32), etiquetas

    def make_ds(x, y, training=False):
        """El mismo pipeline del notebook 01: lotes de 28x28x1 en float32, entre 0 y 255."""
        ds = tf.data.Dataset.from_tensor_slices((x[..., np.newaxis], y))
        if training:
            ds = ds.shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.batch(BATCH_SIZE).map(_a_float32, num_parallel_calls=AUTOTUNE)
        if not training:
            ds = ds.cache()
        return ds.prefetch(AUTOTUNE)

    DS = {"train": make_ds(x_train, y_train, training=True),
          "val": make_ds(x_val, y_val),
          "test": make_ds(x_test, y_test)}

    if N_BUSQUEDA is not None:
        idx, _ = train_test_split(np.arange(len(y_train)), train_size=N_BUSQUEDA,
                                  random_state=SEED, stratify=y_train)
        ds_busqueda = make_ds(x_train[idx], y_train[idx], training=True)
    else:
        ds_busqueda = DS["train"]

## 2. Arquitecturas

Las dos redes se adaptan igual que VGG16 y MobileNetV2: entrada gris de 28×28×1, aumentación opcional,
redimensionamiento bilineal a 96×96, el canal gris copiado tres veces, backbone preentrenado en
ImageNet sin su cabeza, `GlobalAveragePooling2D`, dropout y una capa softmax de 10 clases.

Una diferencia práctica: en Keras, EfficientNetV2 y ConvNeXt traen el preprocesamiento de ImageNet
dentro del modelo (`include_preprocessing=True`) y esperan píxeles entre 0 y 255, que es justo lo que
entrega nuestro pipeline. No hace falta la capa de `Rescaling` que sí necesitaban VGG16 y MobileNetV2.

Para el fine-tuning descongelamos las últimas 20 capas del backbone, como en MobileNetV2, con las
BatchNormalization congeladas. En ConvNeXt eso corresponde a los últimos bloques de la etapa final.

In [ ]:
CLASES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
          "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
NOMBRE = {"cnn": "CNN desde cero", "vgg16": "VGG16", "mobilenetv2": "MobileNetV2",
          "efficientnetv2b0": "EfficientNetV2-B0", "convnext_tiny": "ConvNeXt-Tiny"}
CAPAS_FINE_TUNING = {"efficientnetv2b0": 20, "convnext_tiny": 20}

if ENTRENAR:
    BACKBONES = {
        "efficientnetv2b0": tf.keras.applications.EfficientNetV2B0,
        "convnext_tiny": tf.keras.applications.ConvNeXtTiny,
    }

    def make_augmentation(name):
        return tf.keras.Sequential([
            tf.keras.layers.RandomTranslation(0.08, 0.08, fill_mode="nearest", seed=SEED),
            tf.keras.layers.RandomRotation(0.06, fill_mode="nearest", seed=SEED),
        ], name=name)

    def moderna(cfg, family):
        inputs = tf.keras.Input(shape=(28, 28, 1), name="fashion_mnist_input")
        x = inputs
        if cfg["aug"]:
            x = make_augmentation(f"augmentation_{family}")(x)
        x = tf.keras.layers.Resizing(IMG_SIZE, IMG_SIZE, interpolation="bilinear", name="resize")(x)
        x = tf.keras.layers.Concatenate(axis=-1, name="gris_a_rgb")([x, x, x])

        base = BACKBONES[family](include_top=False, weights="imagenet",
                                 input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True)
        base.trainable = False
        x = base(x, training=False)
        x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
        x = tf.keras.layers.Dropout(cfg["drop"], name="dropout")(x)
        outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="classifier",
                                        dtype="float32")(x)
        return tf.keras.Model(inputs, outputs, name=family)

    def backbone_de(model):
        return next(capa for capa in model.layers if isinstance(capa, tf.keras.Model)
                    and not isinstance(capa, tf.keras.Sequential))

    def compile_m(model, learning_rate):
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                      loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    def cbs():
        # Los mismos del notebook 01. Sin checkpoint: EarlyStopping ya devuelve los mejores pesos.
        return [
            tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
        ]

    def n_entrenables(model):
        return int(sum(np.prod(v.shape) for v in model.trainable_weights))

In [ ]:
# Antes de gastar GPU: las dos redes aceptan 28x28x1, devuelven 10 probabilidades y cargan pesos.
if ENTRENAR:
    dummy = tf.zeros((2, 28, 28, 1), dtype=tf.float32)
    for family in MODELOS_BONO:
        tf.keras.backend.clear_session()
        gc.collect()
        m = moderna({"drop": 0.2, "aug": True}, family)
        salida = m(dummy, training=False)
        base = backbone_de(m)
        assert m.input_shape == (None, 28, 28, 1) and tuple(salida.shape) == (2, NUM_CLASSES)
        print(f"{NOMBRE[family]:<18} backbone {base.name:<20} salida {base.output.shape}"
              f" | parámetros {m.count_params():>11,} | entrenables {n_entrenables(m):>7,}")
        print("   capas que se descongelan en el fine-tuning:",
              [c.name for c in base.layers[-CAPAS_FINE_TUNING[family]:]][:3], "...",
              base.layers[-1].name)
    del m

## 3. Búsqueda de hiperparámetros

La misma rejilla que usamos para VGG16 y MobileNetV2, con el mismo presupuesto de épocas: sin
aumentación y dropout 0.2, con aumentación y dropout 0.3, y esa misma con `lr = 1e-4`. Así la
pregunta "¿ayuda la aumentación?" o "¿1e-4 se queda corto?" se responde igual en las cinco redes.

Cada corrida se escribe en `experimentos_bono.csv` al terminar. Si el notebook se vuelve a correr,
las corridas que ya están en ese archivo no se repiten.

In [ ]:
ESPACIO = [
    {"drop": 0.2, "aug": False, "lr": 1e-3},
    {"drop": 0.3, "aug": True, "lr": 1e-3},
    {"drop": 0.3, "aug": True, "lr": 1e-4},
]
ARCH_EXP = RUTA_BONO / "experimentos_bono.csv"
ARCH_HIST_BUSQ = RUTA_BONO / "historias_busqueda_bono.json"

if ENTRENAR:
    rows = pd.read_csv(ARCH_EXP).to_dict("records") if ARCH_EXP.exists() else []
    search_hist = json.loads(ARCH_HIST_BUSQ.read_text()) if ARCH_HIST_BUSQ.exists() else {}
    hechas = {(r["family"], r["run"]) for r in rows}

    for f in MODELOS_BONO:
        for run, cfg in enumerate(ESPACIO):
            if (f, run) in hechas:
                print(f"{f} corrida {run}: ya estaba, se salta")
                continue
            tf.keras.backend.clear_session()
            gc.collect()
            seed_all()
            m = moderna(cfg, f)
            compile_m(m, cfg["lr"])
            t = time.perf_counter()
            h = m.fit(ds_busqueda, validation_data=DS["val"], epochs=EPOCHS_SEARCH,
                      callbacks=cbs(), verbose=2)
            b = int(np.argmin(h.history["val_loss"]))
            rows.append({
                "family": f, "run": run, "config": json.dumps(cfg),
                "best_epoch": b + 1,
                "val_loss": h.history["val_loss"][b],
                "val_accuracy": h.history["val_accuracy"][b],
                "train_accuracy": h.history["accuracy"][b],
                "gap": h.history["accuracy"][b] - h.history["val_accuracy"][b],
                "seconds": time.perf_counter() - t,
                "params": int(m.count_params()),
                "trainable": n_entrenables(m),
            })
            search_hist[f"{f}_{run}"] = {k: [float(v) for v in vs] for k, vs in h.history.items()}
            pd.DataFrame(rows).to_csv(ARCH_EXP, index=False)
            ARCH_HIST_BUSQ.write_text(json.dumps(search_hist))

    display(pd.DataFrame(rows).sort_values(["family", "val_accuracy"], ascending=[True, False]))

## 4. Entrenamiento final, fine-tuning y evaluación sobre test

Por cada red: la mejor configuración de la búsqueda (mayor accuracy de validación, desempate por
menor pérdida) se entrena con las 54 000 imágenes, luego se descongelan las últimas 20 capas del
backbone con el learning rate dividido por diez, y al final se evalúa **una sola vez** sobre test.

Cada red deja sus predicciones, métricas, curvas y tiempos en disco antes de pasar a la siguiente.
Si ya existen las predicciones de una red, no se vuelve a entrenar.

In [ ]:
ARCH_HIST = RUTA_BONO / "historias_final_bono.json"
ARCH_TIEMPOS = RUTA_BONO / "tiempos_bono.csv"
ARCH_MET = RUTA_BONO / "metricas_test_bono.csv"

if ENTRENAR:
    search = pd.read_csv(ARCH_EXP)
    best = {f: json.loads(search[search.family == f]
                          .sort_values(["val_accuracy", "val_loss"], ascending=[False, True])
                          .iloc[0].config) for f in MODELOS_BONO}
    print("configuraciones elegidas:", json.dumps(best, indent=2))

    hist = json.loads(ARCH_HIST.read_text()) if ARCH_HIST.exists() else {}
    tiempos = pd.read_csv(ARCH_TIEMPOS).to_dict("records") if ARCH_TIEMPOS.exists() else []
    metricas = pd.read_csv(ARCH_MET).to_dict("records") if ARCH_MET.exists() else []

    for f in MODELOS_BONO:
        arch_pred = RUTA_BONO / f"predicciones_{f}.npz"
        if arch_pred.exists():
            print(f"{f}: ya tiene predicciones sobre test, se salta")
            continue
        cfg = best[f]
        tf.keras.backend.clear_session()
        gc.collect()
        seed_all()
        m = moderna(cfg, f)
        compile_m(m, cfg["lr"])
        t = time.perf_counter()
        h = m.fit(DS["train"], validation_data=DS["val"], epochs=EPOCHS_FINAL,
                  callbacks=cbs(), verbose=2)
        seg_base = time.perf_counter() - t
        historia = {k: [float(v) for v in vs] for k, vs in h.history.items()}
        historia["epocas_base"] = len(h.history["loss"])

        n = CAPAS_FINE_TUNING[f]
        base = backbone_de(m)
        base.trainable = True
        for capa in base.layers[:-n]:
            capa.trainable = False
        for capa in base.layers:
            if isinstance(capa, tf.keras.layers.BatchNormalization):
                capa.trainable = False
        compile_m(m, cfg["lr"] / 10)
        t = time.perf_counter()
        h = m.fit(DS["train"], validation_data=DS["val"], epochs=EPOCHS_FT,
                  callbacks=cbs(), verbose=2)
        seg_ft = time.perf_counter() - t
        for k, v in h.history.items():
            historia.setdefault(k, []).extend(float(x) for x in v)

        # Test, una sola vez.
        pr = m.predict(DS["test"], verbose=0)
        yp = pr.argmax(1)
        p, r, f1m, _ = precision_recall_fscore_support(y_test, yp, average="macro", zero_division=0)
        metricas = [x for x in metricas if x["Modelo"] != f] + [{
            "Modelo": f,
            "Accuracy": accuracy_score(y_test, yp),
            "LogLoss": log_loss(y_test, pr, labels=list(range(NUM_CLASSES))),
            "Precision_macro": p, "Recall_macro": r, "F1_macro": f1m,
            "Parametros": int(m.count_params()),
            "Entrenables": n_entrenables(m),
        }]
        print(f"\n{NOMBRE[f]} en test: accuracy {accuracy_score(y_test, yp):.4f} | F1 macro {f1m:.4f}\n")

        hist[f] = historia
        tiempos = [x for x in tiempos if x["family"] != f] + [
            {"family": f, "stage": "base", "seconds": seg_base},
            {"family": f, "stage": f"fine_tune_{n}", "seconds": seg_ft}]
        pd.DataFrame(classification_report(y_test, yp, target_names=CLASES, output_dict=True)).T.to_csv(
            RUTA_BONO / f"reporte_{f}.csv")
        pd.DataFrame(metricas).to_csv(ARCH_MET, index=False)
        pd.DataFrame(tiempos).to_csv(ARCH_TIEMPOS, index=False)
        ARCH_HIST.write_text(json.dumps(hist))
        np.savez_compressed(arch_pred, y_test=y_test, probs=pr.astype("float32"))

        del m
        gc.collect()

    entorno = {
        "tensorflow": tf.__version__,
        "gpu": tf.config.experimental.get_device_details(tf.config.list_physical_devices("GPU")[0])
               .get("device_name") if HAY_GPU else None,
        "presupuesto": PRESUPUESTO, "precision": tf.keras.mixed_precision.global_policy().name,
        "seed": SEED, "img_size": IMG_SIZE, "batch_size": BATCH_SIZE,
        "imagenes_busqueda": N_BUSQUEDA or 54000, "epochs_search": EPOCHS_SEARCH,
        "epochs_final": EPOCHS_FINAL, "epochs_fine_tuning": EPOCHS_FT,
        "capas_fine_tuning": CAPAS_FINE_TUNING, "mejores_config": best,
    }
    (RUTA_BONO / "entorno_bono.json").write_text(json.dumps(entorno, indent=2, ensure_ascii=False),
                                                encoding="utf-8")
    print("GPU usada:", entorno["gpu"])
    if entorno["gpu"] and "T4" not in entorno["gpu"]:
        print("OJO: el notebook 01 corrió en una T4. Con otra GPU los tiempos no son comparables.")

## 5. Figuras y tablas

De aquí para abajo solo se leen archivos: los del notebook `01` en `results/punto2/` y los del bono en
`results/punto2/bono/`. Se mantienen el estilo, los colores y los marcadores de las figuras del
notebook `02`, y se agregan dos colores nuevos para las redes modernas. Los cinco pasan la revisión de
contraste y daltonismo en todos los pares, y cada red tiene además su propio marcador para que se
distingan impresas en gris.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from scipy.stats import binomtest

BASE3 = ["cnn", "vgg16", "mobilenetv2"]
TODAS = BASE3 + MODELOS_BONO

COLOR = {"cnn": "#2a78d6", "vgg16": "#eb6834", "mobilenetv2": "#1baf7a",
         "efficientnetv2b0": "#e0b000", "convnext_tiny": "#4a3aa7"}
MARCA = {"cnn": "o", "vgg16": "s", "mobilenetv2": "^", "efficientnetv2b0": "D", "convnext_tiny": "v"}
TINTA, TINTA_SUAVE, MALLA = "#111111", "#52514e", "#e6e5e1"

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight",
    "figure.facecolor": "white", "axes.facecolor": "white",
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans"],
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "legend.fontsize": 8.5, "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "axes.titleweight": "semibold", "axes.titlelocation": "left", "axes.titlepad": 10,
    "axes.edgecolor": MALLA, "axes.linewidth": 0.8,
    "axes.labelcolor": TINTA_SUAVE, "text.color": TINTA,
    "xtick.color": TINTA_SUAVE, "ytick.color": TINTA_SUAVE,
    "xtick.major.size": 0, "ytick.major.size": 0,
    "axes.grid": True, "grid.color": MALLA, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 1.8,
})
AZULES = LinearSegmentedColormap.from_list(
    "azules", ["#ffffff", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#104281"])


def leyenda_figura(fig, familias, y=-0.02, ncol=None):
    marcas = [Line2D([], [], color=COLOR[f], marker=MARCA[f], linestyle="none", markersize=7,
                     markeredgecolor="white", markeredgewidth=1.2, label=NOMBRE[f])
              for f in familias]
    fig.legend(handles=marcas, loc="lower center", bbox_to_anchor=(0.5, y),
               ncol=ncol or len(familias), handletextpad=0.4, columnspacing=2.0)


def leer_csv(ruta):
    if not ruta.exists():
        print("  falta", ruta.relative_to(RAIZ))
        return None
    return pd.read_csv(ruta)


# Resultados de las tres redes del notebook 01.
met = leer_csv(RUTA_P2 / "metricas_test.csv")
tiempos = leer_csv(RUTA_P2 / "tiempos.csv")
pred01 = np.load(RUTA_P2 / "predicciones_test.npz")
y_test = pred01["y_test"]
probs = {f: pred01[f"probs_{f}"] for f in BASE3}

# Resultados del bono.
met_b = leer_csv(RUTA_BONO / "metricas_test_bono.csv")
tiempos_b = leer_csv(RUTA_BONO / "tiempos_bono.csv")
exp_b = leer_csv(RUTA_BONO / "experimentos_bono.csv")
hist_b = json.loads((RUTA_BONO / "historias_final_bono.json").read_text())
for f in MODELOS_BONO:
    arch = RUTA_BONO / f"predicciones_{f}.npz"
    if arch.exists():
        datos = np.load(arch)
        assert (datos["y_test"] == y_test).all(), "el test del bono no es el mismo del notebook 01"
        probs[f] = datos["probs"]

PRESENTES = [f for f in TODAS if f in probs]
preds = {f: probs[f].argmax(1) for f in PRESENTES}
met_all = pd.concat([met, met_b], ignore_index=True).set_index("Modelo").loc[PRESENTES]
seg_all = pd.concat([tiempos, tiempos_b]).groupby("family").seconds.sum()
print("modelos disponibles:", [NOMBRE[f] for f in PRESENTES])

## 6. Intervalos de confianza y comparaciones pareadas

El intervalo de confianza del accuracy se calcula igual que en el notebook `01` (bootstrap sobre las
10 000 imágenes de test, 1 000 remuestreos). Pero comparar intervalos por separado es conservador:
las cinco redes se evalúan sobre las **mismas** imágenes, así que conviene una comparación pareada.
Hacemos dos:

- **diferencia de accuracy con bootstrap pareado**: en cada remuestreo se toman las mismas imágenes
  para los dos modelos, y se reporta el intervalo del 95 % de la diferencia;
- **prueba de McNemar exacta**, que solo mira las imágenes donde un modelo acierta y el otro no.

Comparamos cada red contra la mejor de las cinco.

In [ ]:
rng = np.random.default_rng(SEED)
B = N_BOOTSTRAP
indices = rng.integers(0, len(y_test), size=(B, len(y_test)))
acierto = {f: (preds[f] == y_test).astype(float) for f in PRESENTES}
boot = {f: acierto[f][indices].mean(axis=1) for f in PRESENTES}

ic = pd.DataFrame({
    "Accuracy": {f: acierto[f].mean() for f in PRESENTES},
    "IC95_inf": {f: np.percentile(boot[f], 2.5) for f in PRESENTES},
    "IC95_sup": {f: np.percentile(boot[f], 97.5) for f in PRESENTES},
})
# Para las tres redes del notebook 01 se conservan los intervalos que ya van en el informe.
boot01 = leer_csv(RUTA_P2 / "bootstrap.csv")
if boot01 is not None:
    ic.loc[boot01.Modelo, ["IC95_inf", "IC95_sup"]] = boot01.set_index("Modelo")[["IC95_inf", "IC95_sup"]].values

MEJOR = ic.Accuracy.idxmax()
pares = []
for f in PRESENTES:
    if f == MEJOR:
        continue
    dif = boot[MEJOR] - boot[f]
    solo_mejor = int(((preds[MEJOR] == y_test) & (preds[f] != y_test)).sum())
    solo_otro = int(((preds[MEJOR] != y_test) & (preds[f] == y_test)).sum())
    pares.append({
        "Modelo": f,
        "Diferencia": acierto[MEJOR].mean() - acierto[f].mean(),
        "Dif_IC95_inf": np.percentile(dif, 2.5),
        "Dif_IC95_sup": np.percentile(dif, 97.5),
        "Solo_acierta_mejor": solo_mejor,
        "Solo_acierta_otro": solo_otro,
        "p_McNemar": binomtest(solo_mejor, solo_mejor + solo_otro, 0.5).pvalue,
    })
pares = pd.DataFrame(pares).set_index("Modelo")
print("mejor red en test:", NOMBRE[MEJOR])
display(ic.round(4))
display(pares.round(4))

## 7. La búsqueda de hiperparámetros de las redes modernas

Mismo formato que la figura de búsqueda del notebook `02`: a la izquierda qué tan lejos llegó cada
configuración, a la derecha cuánto de eso es sobreajuste.

In [ ]:
def etiqueta_config(cfg):
    partes = [f"drop {cfg['drop']:g}", "aug" if cfg["aug"] else "sin aug", f"lr {cfg['lr']:g}"]
    return " · ".join(partes)


if exp_b is not None:
    e = exp_b.copy()
    e["etiqueta"] = e.config.map(lambda s: etiqueta_config(json.loads(s)))
    fams = [f for f in MODELOS_BONO if f in set(e.family)]
    lim = (e.val_accuracy.min() - 0.03, e.val_accuracy.max() + 0.025)

    fig = plt.figure(figsize=(11.5, 4.2))
    rejilla = fig.add_gridspec(len(fams), 2, width_ratios=[1.35, 1], hspace=0.75, wspace=0.32)
    a2 = fig.add_subplot(rejilla[:, 1])
    for k, f in enumerate(fams):
        a = fig.add_subplot(rejilla[k, 0])
        bloque = e[e.family == f].sort_values("val_accuracy").reset_index(drop=True)
        y = np.arange(len(bloque))
        a.hlines(y, lim[0], bloque.val_accuracy, color=COLOR[f], linewidth=1.4, alpha=0.4)
        a.scatter(bloque.val_accuracy, y, color=COLOR[f], marker=MARCA[f], s=48, zorder=3,
                  edgecolor="white", linewidth=1.2)
        a.set_yticks(y, bloque.etiqueta)
        a.set_ylim(-0.7, len(bloque) - 0.3)
        a.set_xlim(*lim)
        a.set_title(NOMBRE[f], color=TINTA, fontsize=9)
        a.grid(axis="y", visible=False)
        if k < len(fams) - 1:
            a.set_xticklabels([])
        else:
            a.set_xlabel("accuracy de validación (mejor época)")
        g = bloque.sort_values(["val_accuracy", "val_loss"], ascending=[False, True]).iloc[0]
        a.annotate(f"{g.val_accuracy:.4f}", (g.val_accuracy, g.name), xytext=(8, 0),
                   textcoords="offset points", va="center", fontsize=8.5, fontweight="semibold")
        if k == 0:
            a.text(0, 1.45, "a. Qué tan lejos llegó cada configuración", transform=a.transAxes,
                   fontsize=10, fontweight="semibold")
        a2.scatter(bloque.gap, bloque.val_accuracy, s=60, marker=MARCA[f], color=COLOR[f],
                   edgecolor="white", linewidth=1.2, zorder=3)
    a2.axvline(0, color=MALLA, linewidth=0.8, zorder=1)
    a2.set_xlabel("brecha entrenamiento − validación")
    a2.set_ylabel("accuracy de validación")
    a2.set_title("b. Rendimiento contra sobreajuste")
    a2.margins(x=0.15, y=0.15)
    leyenda_figura(fig, fams, y=-0.06)
    fig.savefig(RUTA_BONO / "fig_bono_busqueda.png")
    plt.show()

## 8. Curvas de entrenamiento

Línea gruesa: validación. Línea tenue: entrenamiento. La línea vertical punteada marca dónde empieza
el fine-tuning de cada red.

In [ ]:
fams = [f for f in MODELOS_BONO if f in hist_b]
fig, ejes = plt.subplots(1, 2, figsize=(11, 4.2))
for a, metrica, titulo in [(ejes[0], "accuracy", "a. Accuracy"), (ejes[1], "loss", "b. Pérdida")]:
    for f in fams:
        h = hist_b[f]
        epocas = np.arange(1, len(h[metrica]) + 1)
        a.plot(epocas, h[metrica], color=COLOR[f], linewidth=1.0, alpha=0.38)
        a.plot(epocas, h["val_" + metrica], color=COLOR[f], linewidth=2.0)
        a.axvline(h["epocas_base"] + 0.5, color=COLOR[f], linewidth=1.0, linestyle=":")
    a.set_xlabel("época"); a.set_title(titulo)
    a.margins(x=0.10)

# Si los dos valores finales quedan muy cerca, se separan las etiquetas en vertical.
ultima = max(len(hist_b[f]["val_accuracy"]) for f in fams)
finales = sorted(fams, key=lambda f: hist_b[f]["val_accuracy"][-1])
juntas = len(fams) == 2 and abs(hist_b[fams[0]]["val_accuracy"][-1] - hist_b[fams[1]]["val_accuracy"][-1]) < 0.004
for k, f in enumerate(finales):
    serie = hist_b[f]["val_accuracy"]
    dy = (-6 if k == 0 else 6) if juntas else 0
    ejes[0].annotate(f"{serie[-1]:.3f}", (ultima, serie[-1]), xytext=(10, dy),
                     textcoords="offset points", va="center", fontsize=8.5, fontweight="semibold",
                     color=COLOR[f])
ejes[0].set_xlim(right=ultima * 1.18)
ejes[0].set_ylabel("accuracy"); ejes[1].set_ylabel("pérdida")
fig.text(0.5, -0.03, "línea gruesa: validación · línea tenue: entrenamiento · punteada: inicio del fine-tuning",
         ha="center", fontsize=8, color=TINTA_SUAVE)
leyenda_figura(fig, fams, y=-0.11)
fig.savefig(RUTA_BONO / "fig_bono_curvas.png")
plt.show()

## 9. Las cinco redes frente a frente

La figura central del bono. Mismo planteamiento que `fig_comparacion.png` del notebook `02`, ahora
con las cinco redes: accuracy contra parámetros entrenables y contra segundos de GPU, con el intervalo
de confianza del 95 %. El panel **c** es la comparación pareada contra la mejor red: si el intervalo
de la diferencia cruza el cero, la ventaja no es concluyente.

In [ ]:
fig = plt.figure(figsize=(12, 7.6))
rejilla = fig.add_gridspec(2, 2, height_ratios=[1, 0.8], hspace=0.5, wspace=0.28)
a1, a2 = fig.add_subplot(rejilla[0, 0]), fig.add_subplot(rejilla[0, 1])
a3 = fig.add_subplot(rejilla[1, :])

for a, serie, etiqueta_x, titulo, log in [
    (a1, met_all.Entrenables, "parámetros entrenables (escala log)", "a. Accuracy contra tamaño del modelo", True),
    (a2, seg_all, "segundos de entrenamiento (búsqueda aparte)", "b. Accuracy contra tiempo de GPU", False),
]:
    for f in PRESENTES:
        acc = ic.loc[f, "Accuracy"]
        a.errorbar(serie[f], acc, yerr=[[acc - ic.loc[f, "IC95_inf"]], [ic.loc[f, "IC95_sup"] - acc]],
                   fmt=MARCA[f], color=COLOR[f], markersize=9, markeredgecolor="white",
                   markeredgewidth=1.6, capsize=4, elinewidth=1.4, zorder=3)
    if log:
        a.set_xscale("log")
    else:
        a.set_xlim(left=0)
    a.set_xlabel(etiqueta_x); a.set_ylabel("accuracy en test"); a.set_title(titulo)
    a.margins(x=0.15, y=0.2)

orden = pares.sort_values("Diferencia").index
y = np.arange(len(orden))
for k, f in enumerate(orden):
    fila = pares.loc[f]
    a3.errorbar(fila.Diferencia * 100, k,
                xerr=[[(fila.Diferencia - fila.Dif_IC95_inf) * 100], [(fila.Dif_IC95_sup - fila.Diferencia) * 100]],
                fmt=MARCA[f], color=COLOR[f], markersize=9, markeredgecolor="white",
                markeredgewidth=1.6, capsize=4, elinewidth=1.4, zorder=3)
    texto = f"p = {fila.p_McNemar:.3f}" if fila.p_McNemar >= 0.001 else "p < 0.001"
    a3.annotate(texto, (fila.Dif_IC95_sup * 100, k), xytext=(8, 0), textcoords="offset points",
                va="center", fontsize=8, color=TINTA_SUAVE)
a3.axvline(0, color=TINTA_SUAVE, linewidth=1.0, zorder=1)
a3.set_yticks(y, [NOMBRE[f] for f in orden])
a3.set_ylim(-0.6, len(orden) - 0.4)
a3.grid(axis="y", visible=False)
a3.set_xlim(left=min(-0.3, pares.Dif_IC95_inf.min() * 100 - 0.3))
a3.margins(x=0.12)
a3.set_xlabel(f"puntos de accuracy que {NOMBRE[MEJOR]} le saca a cada red (IC 95 %, bootstrap pareado)")
a3.set_title(f"c. Diferencia pareada contra {NOMBRE[MEJOR]} · p de la prueba de McNemar a la derecha")

leyenda_figura(fig, PRESENTES, y=0.0)
fig.savefig(RUTA_BONO / "fig_bono_comparacion.png")
plt.show()

## 10. Dónde se equivocan las redes modernas

Matrices de confusión normalizadas por fila, con la misma escala y la misma rampa de color que las del
notebook `02`, para poder ponerlas lado a lado. Después, el F1 de cada clase para las cinco redes.

In [ ]:
fams = [f for f in MODELOS_BONO if f in preds]
fig, ejes = plt.subplots(1, len(fams), figsize=(5.1 * len(fams), 5.2))
ejes = np.atleast_1d(ejes)
for k, (a, f) in enumerate(zip(ejes, fams)):
    cm = confusion_matrix(y_test, preds[f], normalize="true") * 100
    imagen = a.imshow(cm, cmap=AZULES, vmin=0, vmax=100, interpolation="nearest")
    for i in range(10):
        for j in range(10):
            if cm[i, j] >= 1:
                a.text(j, i, f"{cm[i, j]:.0f}", ha="center", va="center", fontsize=7,
                       color="white" if cm[i, j] > 55 else TINTA)
    a.set_xticks(range(10), CLASES, rotation=90)
    a.set_yticks(range(10), CLASES if k == 0 else [""] * 10)
    a.set_title(f"{NOMBRE[f]} · {(preds[f] == y_test).mean():.2%}")
    a.set_xlabel("predicción")
    a.grid(False)
ejes[0].set_ylabel("clase real")
barra = fig.colorbar(imagen, ax=ejes, fraction=0.016, pad=0.015)
barra.set_label("% de la clase real", fontsize=8.5)
barra.outline.set_visible(False)
fig.savefig(RUTA_BONO / "fig_bono_confusion.png")
plt.show()

for f in fams:
    cm = confusion_matrix(y_test, preds[f])
    np.fill_diagonal(cm, 0)
    top = np.dstack(np.unravel_index(np.argsort(cm.ravel())[::-1][:5], cm.shape))[0]
    print(NOMBRE[f] + ":", " | ".join(f"{CLASES[i]} -> {CLASES[j]}: {cm[i, j]}" for i, j in top))

In [ ]:
f1 = pd.DataFrame({f: f1_score(y_test, preds[f], average=None) for f in PRESENTES}, index=CLASES)
f1 = f1.loc[f1.mean(axis=1).sort_values().index]

fig, a = plt.subplots(figsize=(8.6, 5.6))
y = np.arange(len(f1))
a.hlines(y, f1.min(axis=1), f1.max(axis=1), color=MALLA, linewidth=2.5, zorder=1)
corrimiento = dict(zip(PRESENTES, np.linspace(0.32, -0.32, len(PRESENTES))))
for f in PRESENTES:
    a.scatter(f1[f], y + corrimiento[f], marker=MARCA[f], s=52, color=COLOR[f],
              edgecolor="white", linewidth=1.2, zorder=3)
a.set_yticks(y, f1.index)
a.set_ylim(-0.6, len(f1) - 0.4)
a.set_xlabel("F1 por clase")
a.set_title("De la clase más difícil a la más fácil, cinco redes")
a.grid(axis="y", visible=False)
a.margins(x=0.08)

# Se rotula solo la clase más difícil: el peor y el mejor valor.
fila = f1.iloc[0]
for f, dy in [(fila.idxmin(), 10), (fila.idxmax(), 10)]:
    a.annotate(f"{fila[f]:.2f}", (fila[f], corrimiento[f]), xytext=(0, dy), textcoords="offset points",
               ha="center", va="bottom", fontsize=8.5, fontweight="semibold")

leyenda_figura(fig, PRESENTES, y=-0.08, ncol=3)
fig.savefig(RUTA_BONO / "fig_bono_f1_clase.png")
plt.show()
display(f1.round(4))

## 11. Tablas para el informe

Igual que en el notebook `02`: cada tabla en `.csv`, `.tex` y `.md`.

In [ ]:
def exportar(df, nombre, titulo, indice=False):
    df.to_csv(RUTA_BONO / f"tabla_{nombre}.csv", index=indice)
    (RUTA_BONO / f"tabla_{nombre}.tex").write_text(
        df.to_latex(index=indice, escape=True, caption=titulo, label=f"tab:{nombre}",
                    float_format="%.4f"), encoding="utf-8")
    try:
        texto = f"**{titulo}**\n\n" + df.to_markdown(index=indice, floatfmt=".4f")
    except ImportError:
        print("  falta la librería tabulate, no se genera el .md de", nombre)
    else:
        (RUTA_BONO / f"tabla_{nombre}.md").write_text(texto, encoding="utf-8")
    print("tabla_" + nombre, "->", len(df), "filas")
    display(df)


comparativa = met_all.join(ic[["IC95_inf", "IC95_sup"]]).join(seg_all.rename("seconds")).reset_index()
comparativa["IC 95%"] = comparativa.apply(lambda r: f"[{r.IC95_inf:.4f}, {r.IC95_sup:.4f}]", axis=1)
comparativa["Modelo"] = comparativa.Modelo.map(NOMBRE)
comparativa = comparativa[["Modelo", "Accuracy", "IC 95%", "F1_macro", "Precision_macro", "Recall_macro",
                           "LogLoss", "Parametros", "Entrenables", "seconds"]]
comparativa.columns = ["Modelo", "Accuracy", "IC 95%", "F1 macro", "Precisión macro", "Recall macro",
                       "Log-loss", "Parámetros", "Entrenables", "Tiempo (s)"]
exportar(comparativa, "bono_comparativa",
         "Comparación de las cinco redes sobre el conjunto de test (10 000 imágenes)")

tabla_pares = pares.reset_index()
tabla_pares["IC 95% diferencia"] = tabla_pares.apply(
    lambda r: f"[{r.Dif_IC95_inf:.4f}, {r.Dif_IC95_sup:.4f}]", axis=1)
tabla_pares["Modelo"] = tabla_pares.Modelo.map(NOMBRE)
tabla_pares = tabla_pares[["Modelo", "Diferencia", "IC 95% diferencia", "Solo_acierta_mejor",
                           "Solo_acierta_otro", "p_McNemar"]]
tabla_pares.columns = ["Modelo", f"Diferencia ({NOMBRE[MEJOR]} − modelo)", "IC 95% diferencia",
                       f"Solo acierta {NOMBRE[MEJOR]}", "Solo acierta el modelo", "p (McNemar)"]
exportar(tabla_pares, "bono_pareada", f"Comparación pareada de cada red contra {NOMBRE[MEJOR]} sobre test")

if exp_b is not None:
    busqueda = exp_b.copy()
    busqueda["Configuración"] = busqueda.config.map(lambda s: etiqueta_config(json.loads(s)))
    busqueda["Modelo"] = busqueda.family.map(NOMBRE)
    busqueda = (busqueda[["Modelo", "Configuración", "best_epoch", "val_accuracy", "val_loss", "gap", "seconds"]]
                .sort_values(["Modelo", "val_loss"]))
    busqueda.columns = ["Modelo", "Configuración", "Mejor época", "Accuracy val.", "Pérdida val.",
                        "Brecha", "Tiempo (s)"]
    exportar(busqueda, "bono_busqueda", "Búsqueda de hiperparámetros de EfficientNetV2-B0 y ConvNeXt-Tiny")

por_clase = f1.copy()
por_clase.columns = [NOMBRE[c] for c in por_clase.columns]
por_clase.index.name = "Clase"
exportar(por_clase.reset_index(), "bono_f1_por_clase", "F1 por clase de las cinco redes")

## 12. Observaciones

*(Se llenan con los números de la corrida.)*

- **Desempeño**: ¿alguna red moderna supera a VGG16 de forma concluyente según el panel c y McNemar,
  o las diferencias quedan dentro del ruido del test?
- **Costo**: ¿qué se paga en parámetros y en segundos de GPU por cada punto de accuracy? Comparar
  EfficientNetV2-B0 contra MobileNetV2 (misma familia de bloques) y ConvNeXt-Tiny contra VGG16
  (las dos más pesadas).
- **Búsqueda**: ¿se repite lo del notebook `01`, que la aumentación no ayudó con 12 épocas y que
  `lr = 1e-4` se quedó corto?
- **Errores**: ¿las redes modernas se equivocan en las mismas clases (*Shirt* contra *T-shirt/top*,
  *Pullover* y *Coat*) o cambian el patrón?
- **Limitación**: las redes se usan a 96×96, muy por debajo de la resolución con la que se
  preentrenaron (224×224 en los dos casos); con imágenes de 28×28 subirlas más
  solo interpola píxeles y encarece el entrenamiento.

In [ ]:
# Para bajar todo el bono de un solo golpe desde el panel de archivos de Colab.
if EN_COLAB:
    !cd /content/taller1-deep-learning && zip -qr /content/punto2_bono.zip results/punto2/bono
    !ls -lh /content/punto2_bono.zip
print("archivos en results/punto2/bono/:")
for p in sorted(RUTA_BONO.iterdir()):
    print(f"  {p.name:<36} {p.stat().st_size / 1024:8.1f} KB")